# Laboratorio 7 - Carga, armonización y calidad de datos

Esta sección prepara las bases de Personas de ENEIC para el análisis posterior. La conversión inicial desde Excel usa pandas; todas las validaciones, filtros y persistencia se realizan con PySpark.

In [11]:
from functools import reduce
from pathlib import Path

import pandas as pd
from pyspark.sql import DataFrame, SparkSession, functions as F

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
spark = SparkSession.builder.appName('DS-LAB07').getOrCreate()
spark.sparkContext.setLogLevel('WARN')

In [12]:
# Funciones de preparación. Todas las transformaciones analíticas se ejecutan en Spark.
RAW_COLUMNS = ['P05D01', 'P02A03', 'P05C07A', 'P05C07B', 'P05H01A', 'P03A03A', 'P05C16', 'DOMINIO', 'OCUPADOS', 'NUM_HOGAR', 'NUM_PERSONA', 'FACTOR', 'ANIO', 'TRIMESTRE']
NUMERIC_COLUMNS = ['P05D01', 'P02A03', 'P05C07A', 'P05C07B', 'P05H01A', 'FACTOR']
CATEGORICAL_COLUMNS = ['P03A03A', 'P05C16', 'DOMINIO', 'OCUPADOS']

def normalize_code(column):
    value = F.trim(column.cast('string'))
    return F.when(value.rlike(r'^[+-]?\d+\.0+$'), F.regexp_replace(value, r'\.0+$', '')).otherwise(value)

def numeric_value(column):
    value = F.regexp_replace(F.trim(column.cast('string')), ',', '')
    return F.when(value.rlike(r'^[+-]?(\d+(\.\d*)?|\.\d+)([eE][+-]?\d+)?$'), value.cast('double'))

def read_personas_excel(path, periodo_archivo, anio_archivo, trimestre_calendario):
    path = Path(path)
    source = pd.read_excel(path, usecols=lambda name: name in RAW_COLUMNS, dtype='string')
    missing = sorted(set(RAW_COLUMNS) - set(source.columns))
    if missing:
        raise ValueError(f'{path.name} no contiene las columnas requeridas: {missing}')
    source = source[RAW_COLUMNS]
    source['archivo_origen'] = path.name
    source['periodo_archivo'] = periodo_archivo
    source['anio_archivo'] = str(anio_archivo)
    source['trimestre_calendario'] = str(trimestre_calendario)
    return spark.createDataFrame(source.fillna(''))

def harmonize_types(frame):
    result = frame
    for name in NUMERIC_COLUMNS:
        result = result.withColumn(name, numeric_value(F.col(name)))
    for name in CATEGORICAL_COLUMNS + ['NUM_HOGAR', 'NUM_PERSONA', 'ANIO', 'TRIMESTRE']:
        result = result.withColumn(name, normalize_code(F.col(name)))
    return (result.withColumn('anio_archivo', F.col('anio_archivo').cast('int'))
        .withColumn('trimestre_calendario', F.col('trimestre_calendario').cast('int'))
        .withColumnRenamed('P05D01', 'salario_mensual').withColumnRenamed('P02A03', 'edad')
        .withColumnRenamed('P05C07A', 'antiguedad_anios').withColumnRenamed('P05C07B', 'antiguedad_meses')
        .withColumnRenamed('P05H01A', 'horas_semanales').withColumnRenamed('P03A03A', 'nivel_educativo')
        .withColumnRenamed('P05C16', 'categoria_ocupacional').withColumnRenamed('DOMINIO', 'dominio')
        .withColumnRenamed('OCUPADOS', 'ocupado'))

def union_frames(frames):
    if not frames:
        raise ValueError('No se proporcionaron DataFrames para unir.')
    return reduce(lambda left, right: left.unionByName(right, allowMissingColumns=True), frames)

def missing_report(frame, columns):
    total = frame.count()
    numeric_types = {'double', 'float', 'int', 'bigint', 'decimal'}
    types = dict(frame.dtypes)
    rows = []
    for name in columns:
        missing = F.col(name).isNull() | (F.trim(F.col(name).cast('string')) == '')
        if any(types[name].startswith(data_type) for data_type in numeric_types):
            missing = missing | F.isnan(F.col(name))
        rows.append((name, frame.where(missing).count()))
    return spark.createDataFrame([(name, missing, 100 * missing / total if total else 0.0) for name, missing in rows], ['variable', 'faltantes', 'porcentaje_faltantes'])

def categorize_values(frame, valid_codes):
    result = frame
    for name in ['nivel_educativo', 'categoria_ocupacional', 'dominio']:
        allowed = valid_codes.get(name)
        if not allowed:
            raise ValueError(f'Faltan códigos validados del diccionario para {name}.')
        result = result.withColumn(name, F.when(F.col(name).isin(*sorted(allowed)), F.col(name)).otherwise(F.lit('DESCONOCIDO')))
    return result

def filter_population(frame):
    stages = [
        ('edad_valida', F.col('edad').isNotNull() & ~F.isnan('edad') & (F.col('edad') >= 15)),
        ('ocupado', F.col('ocupado') == '1'),
        ('asalariado', F.col('categoria_ocupacional').isin('1', '2', '3', '4')),
        ('salario_positivo', F.col('salario_mensual').isNotNull() & ~F.isnan('salario_mensual') & (F.col('salario_mensual') > 0)),
        ('antiguedad_valida', F.col('antiguedad_anios').isNotNull() & ~F.isnan('antiguedad_anios') & (F.col('antiguedad_anios') >= 0) & F.col('antiguedad_meses').isNotNull() & ~F.isnan('antiguedad_meses') & (F.col('antiguedad_meses') >= 0) & (F.col('antiguedad_meses') <= 11) & (F.floor(F.col('antiguedad_meses')) == F.col('antiguedad_meses'))),
        ('antiguedad_no_mayor_edad', (F.col('antiguedad_anios') + F.col('antiguedad_meses') / 12) <= F.col('edad')),
        ('horas_validas', F.col('horas_semanales').isNotNull() & ~F.isnan('horas_semanales') & (F.col('horas_semanales') > 0) & (F.col('horas_semanales') <= 168)),
    ]
    current, report = frame, []
    for stage, condition in stages:
        before = current.count()
        current = current.where(condition)
        after = current.count()
        report.append((stage, before, after, before - after))
    prepared = current.withColumn('antiguedad', F.col('antiguedad_anios') + F.col('antiguedad_meses') / 12)
    return prepared, spark.createDataFrame(report, ['paso', 'registros_antes', 'registros_despues', 'registros_excluidos'])

def counts_by_file(before, after):
    initial = before.groupBy('archivo_origen').count().withColumnRenamed('count', 'antes_filtros')
    final = after.groupBy('archivo_origen').count().withColumnRenamed('count', 'despues_filtros')
    return initial.join(final, 'archivo_origen', 'left').fillna(0).orderBy('archivo_origen')

def duplicate_audit(frame):
    keys = ['periodo_archivo', 'NUM_HOGAR', 'NUM_PERSONA']
    repeated = frame.groupBy(*keys).count().where(F.col('count') > 1)
    repeated_rows = frame.join(repeated.select(*keys), keys, 'inner')
    compared = [name for name in frame.columns if name not in keys]
    signatures = repeated_rows.withColumn('firma_registro', F.sha2(F.concat_ws('||', *[F.coalesce(F.col(name).cast('string'), F.lit('<NULL>')) for name in compared]), 256))
    summary = signatures.groupBy(*keys).agg(F.count('*').alias('filas'), F.countDistinct('firma_registro').alias('versiones_distintas')).withColumn('clasificacion', F.when(F.col('versiones_distintas') == 1, 'repeticion_exacta').otherwise('conflicto'))
    return summary, repeated_rows

## Configuración de archivos

Actualizar las rutas con los nombres reales de los cinco archivos descargados. Cada período se asigna según el archivo de procedencia, no mediante la columna original `TRIMESTRE`.

In [13]:
RAW_DATA_DIR = PROJECT_ROOT / 'data' / 'raw'
DICTIONARY_DIR = PROJECT_ROOT / 'data' / 'diccionarios'
FILES = {
    '2025T1': (RAW_DATA_DIR / 'Personas_ENEIC_T1_2025.xlsx', 2025, 1),
    '2025T2': (RAW_DATA_DIR / 'Personas-ENEIC-T2-2025.xlsx', 2025, 2),
    '2025T3': (RAW_DATA_DIR / 'Base-de-datos-Personas-ENEIC-III-2025.xlsx', 2025, 3),
    '2025T4': (RAW_DATA_DIR / 'Base-de-datos-Personas-ENEIC-IV-2025.xlsx', 2025, 4),
    '2026T1': (RAW_DATA_DIR / 'Base-de-datos-Personas-ENEIC-I-2026.xlsx', 2026, 1),
}
missing_files = [str(path) for path, _, _ in FILES.values() if not path.exists()]
if missing_files:
    raise FileNotFoundError('Actualizar FILES con las rutas reales:\n' + '\n'.join(missing_files))

In [14]:
frames = {
    period: harmonize_types(read_personas_excel(path, period, year, quarter))
    for period, (path, year, quarter) in FILES.items()
}
data_2025_raw = union_frames([frames[p] for p in ('2025T1', '2025T2', '2025T3', '2025T4')])
data_2026_raw = frames['2026T1']
data_2025_raw.printSchema()
data_2025_raw.select('archivo_origen', 'periodo_archivo', 'edad', 'salario_mensual', 'categoria_ocupacional').show(5, truncate=False)

root
 |-- salario_mensual: double (nullable = true)
 |-- edad: double (nullable = true)
 |-- antiguedad_anios: double (nullable = true)
 |-- antiguedad_meses: double (nullable = true)
 |-- horas_semanales: double (nullable = true)
 |-- nivel_educativo: string (nullable = true)
 |-- categoria_ocupacional: string (nullable = true)
 |-- dominio: string (nullable = true)
 |-- ocupado: string (nullable = true)
 |-- NUM_HOGAR: string (nullable = true)
 |-- NUM_PERSONA: string (nullable = true)
 |-- FACTOR: double (nullable = true)
 |-- ANIO: string (nullable = true)
 |-- TRIMESTRE: string (nullable = true)
 |-- archivo_origen: string (nullable = true)
 |-- periodo_archivo: string (nullable = true)
 |-- anio_archivo: integer (nullable = true)
 |-- trimestre_calendario: integer (nullable = true)

+---------------------------+---------------+----+---------------+---------------------+
|archivo_origen             |periodo_archivo|edad|salario_mensual|categoria_ocupacional|
+---------------------

In [15]:
selected_columns = ['salario_mensual', 'edad', 'antiguedad_anios', 'antiguedad_meses', 'horas_semanales', 'nivel_educativo', 'categoria_ocupacional', 'dominio', 'ocupado']
missing_report(data_2025_raw, selected_columns).orderBy('variable').show(truncate=False)

Py4JJavaError: An error occurred while calling o2533.showString.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 5 in stage 59.0 failed 1 times, most recent failure: Lost task 5.0 in stage 59.0 (TID 1238) (DIEGO-LENOVO executor driver): org.apache.spark.SparkException: Python worker failed to connect back.
	at org.apache.spark.api.python.PythonWorkerFactory.createSimpleWorker(PythonWorkerFactory.scala:303)
	at org.apache.spark.api.python.PythonWorkerFactory.create(PythonWorkerFactory.scala:154)
	at org.apache.spark.SparkEnv.createPythonWorker(SparkEnv.scala:182)
	at org.apache.spark.api.python.BasePythonRunner.compute(PythonRunner.scala:330)
	at org.apache.spark.api.python.PythonRDD.compute(PythonRDD.scala:73)
	at org.apache.spark.rdd.RDD.computeOrReadCheckpoint(RDD.scala:374)
	at org.apache.spark.rdd.RDD.iterator(RDD.scala:338)
	at org.apache.spark.rdd.MapPartitionsRDD.compute(MapPartitionsRDD.scala:57)
	at org.apache.spark.rdd.RDD.computeOrReadCheck

## Validación de categorías

Completar las listas con códigos confirmados en los diccionarios de ENEIC. El código educativo `0` debe incluirse porque representa `ninguno`; no es un valor faltante.

In [ ]:
VALID_CODES = {
    'nivel_educativo': {'0', '1', '2', '3', '4', '5', '6', '7'},
    'categoria_ocupacional': {'1', '2', '3', '4', '5', '6', '7', '8', '9'},
    'dominio': {'1', '2', '3'},
}
# Códigos confirmados con los diccionarios de Personas ENEIC descargados del INE.
data_2025_categorized = categorize_values(data_2025_raw, VALID_CODES)
data_2026_categorized = categorize_values(data_2026_raw, VALID_CODES)

In [ ]:
data_2025_prepared, exclusions_2025 = filter_population(data_2025_categorized)
data_2026_prepared, exclusions_2026 = filter_population(data_2026_categorized)

counts_by_file(data_2025_raw, data_2025_prepared).show(truncate=False)
exclusions_2025.show(truncate=False)

In [ ]:
duplicate_summary, duplicate_rows = duplicate_audit(data_2025_raw)
duplicate_summary.groupBy('clasificacion').count().show()
duplicate_summary.orderBy('periodo_archivo', 'NUM_HOGAR', 'NUM_PERSONA').show(20, truncate=False)
# duplicate_rows contiene el detalle para investigar conflictos sin ocultarlos con dropDuplicates().

In [ ]:
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
data_2025_prepared.write.mode('overwrite').parquet(str(PROCESSED_DIR / 'personas_2025_preparado'))
data_2026_prepared.write.mode('overwrite').parquet(str(PROCESSED_DIR / 'personas_2026_t1_preparado'))

## Interpretación requerida

- IV-2025 no puede apilarse por posición porque tiene 302 columnas, mientras los demás archivos tienen 270. `unionByName` evita asociar columnas distintas por error.
- Una pregunta no aplicable significa que, por el flujo de la encuesta, no correspondía responderla. Una respuesta no registrada significa que la pregunta aplicaba, pero no se cuenta con un valor. Ambas situaciones deben documentarse y no interpretarse como el código cero.
- Una misma persona puede aparecer en más de un trimestre debido al diseño longitudinal con rotación. Cada aparición corresponde a una observación de un período distinto, por lo que no es un duplicado que deba eliminarse.
- La base filtrada describe los registros analizados, no a todos los trabajadores de Guatemala. Las métricas obligatorias no son ponderadas; `FACTOR` se conservaría para una estimación poblacional que sí incorporara el diseño muestral.

# Exploración y segmentación (Persona 2)

Esta sección utiliza exclusivamente el conjunto analítico de 2025 generado arriba. Los cálculos se hacen en Spark; únicamente las tablas agregadas y las muestras acotadas se convierten a pandas para graficar.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from pyspark.ml import Pipeline
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
from pyspark.ml.feature import StandardScaler, VectorAssembler
from pyspark.ml.stat import Correlation
from pyspark.sql import Window

# Permite ejecutar la sección después de haber creado los Parquet, incluso en una sesión nueva.
if 'data_2025_prepared' not in globals():
    data_2025_prepared = spark.read.parquet(str(PROJECT_ROOT / 'data' / 'processed' / 'personas_2025_preparado'))

analysis_2025 = data_2025_prepared.cache()
print(f'Observaciones analíticas de 2025: {analysis_2025.count():,}')

## Estadísticas descriptivas

Las estadísticas siguientes no usan `FACTOR`: describen la muestra analítica preparada, no una estimación ponderada de la población.

In [ ]:
NUMERIC_ANALYSIS = ['salario_mensual', 'edad', 'antiguedad', 'horas_semanales']

descriptive_stats = analysis_2025.agg(*[
    expression
    for column in NUMERIC_ANALYSIS
    for expression in (
        F.count(column).alias(f'{column}__n'),
        F.mean(column).alias(f'{column}__media'),
        F.stddev_samp(column).alias(f'{column}__desviacion_estandar'),
        F.min(column).alias(f'{column}__minimo'),
        F.expr(f'percentile_approx({column}, 0.25, 10000)').alias(f'{column}__p25'),
        F.expr(f'percentile_approx({column}, 0.50, 10000)').alias(f'{column}__mediana'),
        F.expr(f'percentile_approx({column}, 0.75, 10000)').alias(f'{column}__p75'),
        F.expr(f'percentile_approx({column}, 0.95, 10000)').alias(f'{column}__p95'),
        F.max(column).alias(f'{column}__maximo'),
    )
]).first().asDict()

statistics_table = spark.createDataFrame([
    (column, *[descriptive_stats[f'{column}__{metric}'] for metric in ['n', 'media', 'mediana', 'desviacion_estandar', 'minimo', 'p25', 'p75', 'p95', 'maximo']])
    for column in NUMERIC_ANALYSIS
], ['variable', 'n', 'media', 'mediana', 'desviacion_estandar', 'minimo', 'p25', 'p75', 'p95', 'maximo'])
statistics_table.show(truncate=False)

## Composición de la muestra

Las etiquetas son códigos ENEIC armonizados en la preparación; `DESCONOCIDO` se mantiene visible para no ocultar registros con códigos ausentes o no reconocidos.

In [ ]:
def category_distribution(frame, column):
    total = frame.count()
    return (frame.groupBy(column).count()
        .withColumn('porcentaje', F.round(F.lit(100.0) * F.col('count') / F.lit(total), 2))
        .orderBy(F.desc('count'), column))

def plot_aggregate(table, category, value, title, ylabel, color='#3977a8'):
    # La tabla agregada tiene pocas filas; es seguro llevarla a pandas para la gráfica.
    pdf = table.toPandas()
    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.bar(pdf[category].astype(str), pdf[value], color=color)
    ax.set(title=title, xlabel=category.replace('_', ' ').title(), ylabel=ylabel)
    ax.tick_params(axis='x', rotation=35)
    plt.tight_layout()
    plt.show()

category_tables = {}
for categorical in ['categoria_ocupacional', 'nivel_educativo', 'dominio']:
    category_tables[categorical] = category_distribution(analysis_2025, categorical)
    category_tables[categorical].show(truncate=False)
    plot_aggregate(category_tables[categorical], categorical, 'count',
                   f'Distribución de {categorical.replace(chr(95), chr(32))}',
                   'Número de observaciones')

## Distribución salarial y comparaciones

La media y la mediana se contrastan porque una media claramente superior a la mediana evidencia asimetría positiva: pocos salarios altos desplazan el promedio. El percentil 95 y el máximo permiten identificar esos extremos sin eliminarlos.

In [ ]:
salary_summary = statistics_table.where(F.col('variable') == 'salario_mensual').first().asDict()
print(f"Media: Q{salary_summary['media']:,.2f}; mediana: Q{salary_summary['mediana']:,.2f}; P95: Q{salary_summary['p95']:,.2f}; máximo: Q{salary_summary['maximo']:,.2f}")
if salary_summary['media'] > salary_summary['mediana']:
    print('Interpretación: la distribución presenta asimetría positiva; los salarios altos elevan la media.')
else:
    print('Interpretación: la media no supera la mediana; revisar el histograma y percentiles antes de concluir sobre la asimetría.')

# Muestra reproducible y acotada solo para visualizar; las tablas y métricas anteriores usan todos los registros.
salary_plot_sample = analysis_2025.select('salario_mensual').orderBy(F.rand(seed=2026)).limit(10000).toPandas()
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.hist(salary_plot_sample['salario_mensual'], bins=60, color='#3977a8', edgecolor='white')
ax.axvline(salary_summary['media'], color='#c23b22', label='Media')
ax.axvline(salary_summary['mediana'], color='#1b7f3a', label='Mediana')
ax.set(title='Distribución del salario mensual (muestra de hasta 10,000)', xlabel='Salario mensual (quetzales)', ylabel='Número de observaciones')
ax.legend()
plt.tight_layout()
plt.show()

def median_salary_by(frame, group):
    return (frame.groupBy(group)
        .agg(F.count('*').alias('observaciones'), F.expr('percentile_approx(salario_mensual, 0.5, 10000)').alias('salario_mediano'))
        .orderBy(F.desc('salario_mediano')))

for group in ['nivel_educativo', 'categoria_ocupacional']:
    comparison = median_salary_by(analysis_2025, group)
    comparison.show(truncate=False)
    plot_aggregate(comparison, group, 'salario_mediano', f'Salario mensual mediano por {group.replace(chr(95), chr(32))}', 'Salario mensual mediano (quetzales)', '#7d5ba6')

In [ ]:
quarter_comparison = (analysis_2025.groupBy('periodo_archivo')
    .agg(F.count('*').alias('observaciones'), F.expr('percentile_approx(salario_mensual, 0.5, 10000)').alias('salario_mediano'))
    .orderBy('periodo_archivo'))
quarter_comparison.show(truncate=False)

quarter_pdf = quarter_comparison.toPandas()
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
axes[0].bar(quarter_pdf['periodo_archivo'], quarter_pdf['observaciones'], color='#3977a8')
axes[0].set(title='Tamaño de la muestra analítica por trimestre', xlabel='Trimestre', ylabel='Observaciones')
axes[1].bar(quarter_pdf['periodo_archivo'], quarter_pdf['salario_mediano'], color='#7d5ba6')
axes[1].set(title='Salario mensual mediano por trimestre', xlabel='Trimestre', ylabel='Quetzales')
plt.tight_layout()
plt.show()
print('Interpretar las diferencias trimestrales junto con el tamaño muestral mostrado; no implican por sí solas cambios poblacionales.')

## Correlación de Pearson

La correlación mide asociación lineal, no causalidad. Se calcula con `VectorAssembler` y `Correlation.corr()` sobre todos los registros preparados.

In [ ]:
correlation_assembler = VectorAssembler(inputCols=NUMERIC_ANALYSIS, outputCol='correlation_features')
correlation_frame = correlation_assembler.transform(analysis_2025).select('correlation_features')
correlation_matrix = Correlation.corr(correlation_frame, 'correlation_features', method='pearson').first()[0].toArray()
correlation_rows = [(NUMERIC_ANALYSIS[i], *[float(correlation_matrix[i, j]) for j in range(len(NUMERIC_ANALYSIS))]) for i in range(len(NUMERIC_ANALYSIS))]
correlation_table = spark.createDataFrame(correlation_rows, ['variable', *NUMERIC_ANALYSIS])
correlation_table.show(truncate=False)

fig, ax = plt.subplots(figsize=(7, 5.8))
image = ax.imshow(correlation_matrix, vmin=-1, vmax=1, cmap='coolwarm')
ax.set_xticks(range(len(NUMERIC_ANALYSIS)), NUMERIC_ANALYSIS, rotation=35, ha='right')
ax.set_yticks(range(len(NUMERIC_ANALYSIS)), NUMERIC_ANALYSIS)
for i in range(len(NUMERIC_ANALYSIS)):
    for j in range(len(NUMERIC_ANALYSIS)):
        ax.text(j, i, f'{correlation_matrix[i, j]:.2f}', ha='center', va='center')
fig.colorbar(image, ax=ax, label='Correlación de Pearson')
ax.set_title('Matriz de correlación: variables numéricas')
plt.tight_layout()
plt.show()
print('Valores cercanos a 1 o -1 indican asociación lineal fuerte; valores cercanos a 0, asociación lineal débil.')

## Segmentación con KMeans

Los clusters se forman con edad, antigüedad y horas semanales, no con salario. Así el salario queda como resultado para interpretar perfiles laborales y no define artificialmente los grupos que luego se comparan por ingreso. Las tres variables se estandarizan porque tienen escalas distintas.

In [ ]:
CLUSTER_FEATURES = ['edad', 'antiguedad', 'horas_semanales']
CLUSTER_SEED = 2026
cluster_assembler = VectorAssembler(inputCols=CLUSTER_FEATURES, outputCol='cluster_features_raw')
cluster_scaler = StandardScaler(inputCol='cluster_features_raw', outputCol='cluster_features', withMean=True, withStd=True)
cluster_preparation = Pipeline(stages=[cluster_assembler, cluster_scaler]).fit(analysis_2025)
cluster_features_frame = cluster_preparation.transform(analysis_2025).cache()

evaluator = ClusteringEvaluator(featuresCol='cluster_features', predictionCol='cluster', metricName='silhouette', distanceMeasure='squaredEuclidean')
kmeans_results = []
clustered_by_k = {}
for k in [2, 3, 4, 5]:
    model = KMeans(k=k, seed=CLUSTER_SEED, featuresCol='cluster_features', predictionCol='cluster').fit(cluster_features_frame)
    clustered = model.transform(cluster_features_frame).cache()
    kmeans_results.append((k, float(evaluator.evaluate(clustered))))
    clustered_by_k[k] = clustered

kmeans_comparison = spark.createDataFrame(kmeans_results, ['k', 'silhouette']).orderBy('k')
kmeans_comparison.show(truncate=False)
best_k = kmeans_comparison.orderBy(F.desc('silhouette'), 'k').first()['k']
print(f'K seleccionado: {best_k}. Se prioriza el mayor silhouette; si la diferencia es mínima, se prefiere el K más pequeño e interpretable.')

kmeans_pdf = kmeans_comparison.toPandas()
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(kmeans_pdf['k'], kmeans_pdf['silhouette'], marker='o', color='#3977a8')
ax.set(title='Comparación de KMeans', xlabel='Número de clusters (K)', ylabel='Silhouette score')
ax.set_xticks([2, 3, 4, 5])
plt.tight_layout()
plt.show()

In [ ]:
best_clustered = clustered_by_k[best_k]
cluster_profile = (best_clustered.groupBy('cluster')
    .agg(
        F.count('*').alias('tamano'),
        *[F.mean(column).alias(f'media_{column}') for column in [*CLUSTER_FEATURES, 'salario_mensual']],
        *[F.expr(f'percentile_approx({column}, 0.5, 10000)').alias(f'mediana_{column}') for column in [*CLUSTER_FEATURES, 'salario_mensual']],
    )
    .orderBy('cluster'))
cluster_profile.show(truncate=False)

# Esta tabla permite describir cada grupo por su tamaño, edad, antigüedad, jornada y salario observado.
# Los nombres descriptivos deben asignarse después de revisar estas medias y medianas, no por el identificador numérico del cluster.
print('Descripción sugerida: identificar para cada cluster si concentra trabajadores de menor/mayor edad, menor/mayor antigüedad y jornadas más cortas/largas; después contrastar su salario mediano.')